# 01 · Predict a held-out single perturbation end-to-end

**Problem.** Given a CRISPRa Perturb-seq screen (Norman et al. 2019, K562), can a GNN over a
biological knowledge graph predict the transcriptome-wide response to a *single* gene activation
it has **never seen during training**?

**Approach.** We run the real `graph-perturb` pipeline:
1. load processed Norman data (`load_norman`), with an offline synthetic fallback;
2. build a GO Biological-Process knowledge graph (`get_graph_source("go_bp")`) over the gene universe;
3. make condition-level splits (`make_splits`) so held-out perturbations never leak into training;
4. train the GraphSAGE GNN briefly (`build_model` + `train_model`);
5. predict one held-out single perturbation and compare predicted vs. true mean delta.

**What to look at.** The predicted-vs-true delta scatter (points on the diagonal = good), the
agreement on the top differentially-expressed (DE) genes, and the printed `compute_metrics`
(Pearson on the delta, MSE, overlap@20).

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.WARNING)
np.random.seed(0)

from graph_perturb.config import DataConfig, SplitConfig, ModelConfig, TrainConfig, EvalConfig
from graph_perturb.data import make_splits
from graph_perturb.data.norman import load_norman, make_synthetic_norman
from graph_perturb.data.dataset import build_dataloaders
from graph_perturb.graphs.registry import get_graph_source
from graph_perturb.models import build_model
from graph_perturb.train import train_model
from graph_perturb.evaluate import evaluate_model, metrics_table
from graph_perturb.metrics import compute_metrics

## 1. Load data (REAL Norman, with offline fallback)

`load_norman` resolves a local `.h5ad` or downloads the scPerturb-harmonized Norman file (guarded by
a free-disk check). On a small / air-gapped box that raises, and we drop to
`make_synthetic_norman` — a tiny but fully-valid dataset built from **real** pathway gene symbols so
the GO/Reactome/STRING backends still find genuine edges. The cell prints which mode it used.

In [ ]:
data_cfg = DataConfig(name="norman", n_top_genes=2000)
try:
    data = load_norman(data_cfg)
    MODE = "REAL Norman Perturb-seq"
except Exception as exc:
    print(f"[fallback] real Norman load failed ({type(exc).__name__}: {exc}\n)")
    data = make_synthetic_norman(n_genes=80, n_conditions=30, seed=0)
    MODE = "SYNTHETIC offline stand-in"

print(f"DATA MODE: {MODE}")
print(f"cells={data.n_cells}  genes={data.n_genes}")
print(f"single conditions example: {[c for c in data.condition_labels() if '+' not in c][:5]}")

## 2. Build the GO knowledge graph + condition-level splits

The graph node order is pinned to `data.gene_names`; `make_splits` holds out a fraction of single
perturbations (`test_single`) and combos (`test_combo`) so generalization is measured over
**unseen perturbations**, not unseen cells.

In [ ]:
source = get_graph_source("go_bp", undirected=True, add_self_loops=True)
graph = source.build(data.gene_names, use_cache=False)
print(graph)

split_cfg = SplitConfig(val_frac=0.1, test_single_frac=0.3, test_combo_frac=0.3, seed=0)
splits = make_splits(data, split_cfg)
print({k: len(v) for k, v in splits.as_dict().items()})
assert splits.test_single, "need at least one held-out single perturbation"

## 3. Train the GraphSAGE GNN (briefly)

Small `hidden_dim` and few epochs so the notebook finishes on CPU. The loop is the real
`train_model`: Adam + grad clipping + validation-loss early stopping.

In [ ]:
model_cfg = ModelConfig(name="gnn", hidden_dim=32, n_layers=2, dropout=0.1, attention_heads=2)
train_cfg = TrainConfig(epochs=6, batch_size=16, lr=1e-3, device="cpu",
                        early_stop_patience=6, log_every=50)

loaders = build_dataloaders(data, graph, splits, train_cfg)
model = build_model(model_cfg, num_genes=data.n_genes, graph=graph)
history = train_model(model, loaders, train_cfg, ckpt_dir=None)
print(f"best epoch={history['best_epoch']}  best val_loss={history['best_val']:.4f}")

## 4. Predict ONE held-out single perturbation

We average the model's per-cell predicted delta over all cells of the chosen held-out condition and
compare it to the true mean delta (`data.delta`).

In [ ]:
import torch
from torch_geometric.loader import DataLoader
from graph_perturb.data import PerturbationDataset

target_cond = splits.test_single[0]
print(f"held-out single perturbation: {target_cond!r}")

ds = PerturbationDataset(data, graph, [target_cond])
loader = DataLoader(ds, batch_size=64, shuffle=False)
model.eval()
preds = []
with torch.no_grad():
    for batch in loader:
        preds.append(model(batch).cpu().numpy())
pred_delta = np.concatenate(preds, axis=0).mean(axis=0)   # [n_genes]
true_delta = np.asarray(data.delta(target_cond))           # [n_genes]
print(f"pred_delta shape={pred_delta.shape}  true_delta shape={true_delta.shape}")

## 5. Visualize: predicted vs. true delta + top DE genes

In [ ]:
gene_names = np.asarray(data.gene_names)
k = min(20, data.n_genes)
true_top = np.argsort(np.abs(true_delta))[-k:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
lim = max(np.abs(true_delta).max(), np.abs(pred_delta).max()) * 1.1
axes[0].scatter(true_delta, pred_delta, s=12, alpha=0.6)
axes[0].plot([-lim, lim], [-lim, lim], "r--", lw=1)
axes[0].set(xlabel="true delta", ylabel="predicted delta",
            title=f"{target_cond}: predicted vs true (n={data.n_genes} genes)")

ypos = np.arange(k)
axes[1].barh(ypos - 0.2, true_delta[true_top], height=0.4, label="true")
axes[1].barh(ypos + 0.2, pred_delta[true_top], height=0.4, label="pred")
axes[1].set_yticks(ypos)
axes[1].set_yticklabels(gene_names[true_top], fontsize=7)
axes[1].invert_yaxis()
axes[1].set(xlabel="delta", title=f"top {k} true DE genes")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. REAL metrics for this condition + the whole held-out split

`compute_metrics` on a single condition gives a per-condition Pearson, MSE and overlap@20;
`evaluate_model` aggregates across the full `test_single` / `test_combo` splits.

In [ ]:
single_m = compute_metrics(pred_delta, true_delta, k=20)
print(f"--- {target_cond} (single condition) ---")
for key, val in single_m.as_dict().items():
    print(f"  {key}: {val}")

eval_cfg = EvalConfig(overlap_k=20, splits=("test_single", "test_combo"))
results = evaluate_model(model, data, graph, splits, eval_cfg)
print("\n--- held-out splits ---")
print(metrics_table(results))

**Takeaway.** Even after a brief CPU run, the GNN recovers the *direction* of the strongest DE
genes for an unseen single perturbation (positive `pearson_delta`, non-trivial `overlap_at_20`).
More epochs and the real Norman data sharpen these. Swap the backend in notebook 02 to see how the
choice of knowledge graph moves these numbers.